<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/cis9557__baseline/blob/main/CIS9557_Module09_NB5_PartV_Composition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS9557 Module 9: Data Visualization Principles  
## Notebook 5: Part V -- Decision Framework: Composition  
<i>a copy of this notebook is available at [bit.ly/cis9557mod09part05](https://bit.ly/cis9557mod09part05)</i>

<b><i><font color=blue>Before proceeding, run the following two code blocks to perform some required housekeeping.</font></i></b>  

Run the cell below to install squarify, which is required for the treemap (Example 19). This is a one-time installation per runtime.

In [ ]:
!pip install squarify -q

Run the cell below to import matplotlib and squarify libraries for chart production.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import squarify
plt.rcParams['figure.dpi']=100
plt.rcParams['font.family']='sans-serif'


Examples 17 and 18 share the same dataset: revenue by product category within each of five sales regions.

In [ ]:
regions_c=['Northeast','Southeast','Midwest','Southwest','West']
cats_c=['Premium','Standard','Economy','Clearance']
rev_c=np.array([[18.2,12.4,15.6,20.1,14.3],
                [22.5,18.3,20.8,19.4,21.2],
                [14.8,16.2,13.5,15.7,17.1],
                [ 5.3, 6.8, 7.2, 4.9, 6.5]])
pal_c=['#2166ac','#4dac26','#d01c8b','#f1a340']


---
## Example 17: Stacked Bar Chart and Its Limitations

**Discussion points**

- The stacked bar chart communicates two things reliably: the aggregate total for each region (total bar height) and the contribution of the baseline category (Premium, blue, bottom segment).
- It communicates interior segment comparisons poorly. Standard, Economy, and Clearance all float at different baseline positions across regions, making their heights visually incomparable.
- The grouped bar chart communicates category comparisons clearly but requires mental addition to reconstruct regional totals.
- Neither chart is unconditionally superior. The choice depends on the analytical question.
- A common error: selecting the stacked bar because it 'shows everything at once.' It shows two things well and several things poorly.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,5))
x5=np.arange(len(regions_c)); width=0.19
bot=np.zeros(len(regions_c))
for i,(cat,row) in enumerate(zip(cats_c,rev_c)):
    axes[0].bar(regions_c,row,bottom=bot,label=cat,color=pal_c[i],alpha=0.85)
    bot+=row
axes[0].set_title('Stacked Bar: Regional Totals Clear,\nInterior Comparisons Difficult',fontsize=11)
axes[0].set_ylabel('Revenue ($ Millions)')
axes[0].legend(fontsize=9,loc='upper right'); axes[0].set_ylim(0,80)
for i,(cat,row) in enumerate(zip(cats_c,rev_c)):
    axes[1].bar(x5+i*width,row,width,label=cat,color=pal_c[i],alpha=0.85)
axes[1].set_xticks(x5+width*1.5); axes[1].set_xticklabels(regions_c)
axes[1].set_title('Grouped Bar: Category Comparisons Clear,\nTotals Require Mental Addition',fontsize=11)
axes[1].set_ylabel('Revenue ($ Millions)')
axes[1].legend(fontsize=9); axes[1].set_ylim(0,30)
plt.suptitle('Example 17: Stacked vs. Grouped Bar -- Same Data, Different Questions',fontsize=12)
plt.tight_layout(); plt.show()


---
## Example 18: 100% Stacked Bar Chart

**Discussion points**

- The 100% stacked bar chart removes the absolute dimension entirely. All bars reach 100%; regional totals are invisible.
- Appropriate when the question is about how the proportional mix differs across regions.
- Inappropriate when absolute totals matter. A region that grew 40% while maintaining the same product mix looks identical to a region that shrank 40% with the same mix.
- The 100% stacked bar and the standard stacked bar answer different questions. Selecting the wrong version is an analytical error.

In [ ]:
prop18=rev_c/rev_c.sum(axis=0)
bot=np.zeros(len(regions_c))
fig,ax=plt.subplots(figsize=(10,5))
for i,(cat,row) in enumerate(zip(cats_c,prop18)):
    ax.bar(regions_c,row,bottom=bot,label=cat,color=pal_c[i],alpha=0.85)
    bot+=row
ax.set_title('Revenue Composition by Product Category Within Each Region',fontsize=13)
ax.set_ylabel('Share of Regional Revenue')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax.legend(fontsize=9,loc='upper right')
plt.tight_layout(); plt.show()


---
## Example 20: Waterfall Chart vs. Stacked Bar Chart

**Discussion points**

- The waterfall chart is the appropriate chart type for any analysis in which the audience needs to understand both the direction and the magnitude of each contributing factor in a bridge from a starting value to an ending value. Variance analysis, period-over-period decomposition, and profit-and-loss bridges are all structurally identical from the chart selection perspective.
- Each intermediate bar is drawn as a floating bar, positioned to show where in the value scale the subtraction occurs. Gross Revenue and Operating Income anchor to the zero baseline. The red-orange bars float between the running total before and after each deduction, making the cumulative progression readable at each step.
- The dashed connector lines make that cumulative progression explicit without adding data encoding. Following the connectors, the audience reads the running total at each step: 185, 113, 75, 54, 40, 29. No mental arithmetic is required.
- The right panel demonstrates the failure mode that the whitepaper describes. Because the stacked bar can only stack values upward, the directional information of each item is stripped out entirely. COGS appears to add to revenue rather than subtract from it. Operating Income appears at the base of the bar as if it generates the expenses above it, which inverts the actual logical and financial relationship.
- The stacked bar total height is 185, which matches Gross Revenue by arithmetic coincidence. This coincidence makes the stacked bar look plausible at a glance and is precisely what makes it a dangerous substitute for the waterfall chart in P&L reporting contexts. The numbers add up correctly; the story they tell is wrong.
- The waterfall chart is the only standard chart type that correctly represents both positive and negative contributions in a sequential decomposition. When the analytical question is about how a starting value was built up or reduced through a sequence of identified factors to arrive at a final value, the waterfall chart is not one option among several. It is the correct choice.
- Discussion: what additional information would an analyst need to extend this chart into a period-over-period variance bridge, and how would the color encoding change if some items were favorable variances rather than deductions?

In [ ]:
# Example 20: Waterfall Chart vs. Stacked Bar Chart
# Left panel: waterfall chart -- Q1 operating income bridge.
# Right panel: stacked bar chart applied to the same data, demonstrating failure.
# The stacked bar cannot represent negative contributions: each item appears as
# an upward addition, stripping out the direction of financial impact entirely.

import matplotlib.patches as mpatches
import numpy as np

# Q1 P&L bridge: Gross Revenue decomposed to Operating Income
labels = [
    'Gross\nRevenue', 'COGS', 'Salaries &\nBenefits', 'Marketing',
    'Technology', 'Facilities', 'Operating\nIncome'
]
values    = [185, -72, -38, -21, -14, -11, 29]
bar_types = ['total', 'sub', 'sub', 'sub', 'sub', 'sub', 'total']
# Verify: 185 - 72 - 38 - 21 - 14 - 11 = 29 (Operating Income)

NAVY  = '#0A1E35'
RED_M = '#d6604d'

# Compute waterfall geometry (spacers, visible bar heights, bar tops)
running  = 0
spacers  = []
heights  = []
bar_tops = []

for v, btype in zip(values, bar_types):
    if btype == 'total' and running == 0:    # starting total
        spacers.append(0)
        heights.append(v)
        running = v
    elif btype == 'total':                   # ending total
        spacers.append(0)
        heights.append(running)
    else:                                    # subtraction bar
        new_running = running + v
        spacers.append(new_running)
        heights.append(abs(v))
        running = new_running
    bar_tops.append(spacers[-1] + heights[-1])

colors = [NAVY if t == 'total' else RED_M for t in bar_types]

fig, (ax_w, ax_s) = plt.subplots(
    1, 2, figsize=(15, 6),
    gridspec_kw={'width_ratios': [2.5, 1.0]}
)

# ---- Left panel: Waterfall chart -------------------------------------------
x     = np.arange(len(labels))
BAR_W = 0.55

# Invisible spacer bars position each visible bar at the correct baseline
ax_w.bar(x, spacers, width=BAR_W, color='none', alpha=0, zorder=1)
ax_w.bar(x, heights, bottom=spacers, width=BAR_W,
         color=colors, alpha=0.88, zorder=3)

# Connector lines: horizontal dashed line at bar_tops[i+1] between each bar pair
# bar_tops[i+1] equals the running total after bar[i], and the top of bar[i+1]
for i in range(len(labels) - 1):
    y_conn = bar_tops[i + 1]
    ax_w.plot(
        [x[i] + BAR_W / 2, x[i + 1] - BAR_W / 2],
        [y_conn, y_conn],
        color='#888888', linewidth=0.8, linestyle='--', zorder=2
    )

# Value labels: inside bar if height >= 15, above bar if smaller
for i, (xi, sp, h, btype) in enumerate(zip(x, spacers, heights, bar_types)):
    lbl = f'${h}M' if btype == 'total' else f'-${h}M'
    if h >= 15:
        ax_w.text(xi, sp + h / 2, lbl, ha='center', va='center',
                  fontsize=8, color='white', fontweight='bold')
    else:
        ax_w.text(xi, sp + h + 3, lbl, ha='center', va='bottom',
                  fontsize=8, color=colors[i], fontweight='bold')

ax_w.set_xticks(x)
ax_w.set_xticklabels(labels, fontsize=8.5)
ax_w.set_ylabel('$ Millions', fontsize=9)
ax_w.set_ylim(0, 215)
ax_w.set_title(
    'Waterfall Chart: Q1 Operating Income Bridge\n'
    'Direction and sequence of each contribution preserved',
    fontsize=10
)
ax_w.spines['top'].set_visible(False)
ax_w.spines['right'].set_visible(False)
ax_w.grid(axis='y', alpha=0.2, linewidth=0.5)
ax_w.axhline(y=0, color='#333333', linewidth=0.5)

# ---- Right panel: Stacked bar (same data, demonstrating failure) -----------
# All magnitudes treated as positive and stacked upward.
# Operating Income at base, expense items above -- logical relationship inverted.
# Stacked bar total = 185 by arithmetic, coincidentally matching Gross Revenue.
stack_labels = [
    'Operating\nIncome', 'Facilities', 'Technology',
    'Marketing', 'Salaries &\nBenefits', 'COGS'
]
stack_vals = [29, 11, 14, 21, 38, 72]
stack_cols = [NAVY] + [RED_M] * 5

bottom = 0
for lbl, val, col in zip(stack_labels, stack_vals, stack_cols):
    ax_s.bar(0, val, bottom=bottom, color=col, alpha=0.88, width=0.5, zorder=3)
    if val >= 14:
        ax_s.text(0, bottom + val / 2, f'{lbl}\n${val}M',
                  ha='center', va='center', fontsize=6.5,
                  color='white', fontweight='bold')
    bottom += val

# Annotation 1: COGS segment (113 to 185, midpoint 149)
ax_s.annotate(
    'COGS appears to ADD\nto revenue, not\nsubtract from it.',
    xy=(0.26, 149), xytext=(0.32, 175),
    fontsize=7.5, color='#333333', style='italic', ha='left',
    arrowprops=dict(arrowstyle='->', color='#555555', lw=0.8)
)

# Annotation 2: Operating Income segment (0 to 29, midpoint 14.5)
ax_s.annotate(
    'Operating Income sits\nat the base, as if it\ncauses the expenses.',
    xy=(0.26, 14.5), xytext=(0.32, 48),
    fontsize=7.5, color='#333333', style='italic', ha='left',
    arrowprops=dict(arrowstyle='->', color='#555555', lw=0.8)
)

ax_s.set_xlim(-0.55, 0.90)
ax_s.set_ylim(0, 215)
ax_s.set_xticks([])
ax_s.set_ylabel('$ Millions', fontsize=9)
ax_s.set_title(
    'Stacked Bar (Same Data)\n'
    'All items appear as upward additions;\n'
    'direction of financial impact lost',
    fontsize=10
)
ax_s.spines['top'].set_visible(False)
ax_s.spines['right'].set_visible(False)
ax_s.grid(axis='y', alpha=0.2, linewidth=0.5)
ax_s.axhline(y=0, color='#333333', linewidth=0.5)

plt.suptitle(
    'Example 20: Waterfall Chart vs. Stacked Bar -- Q1 P&L Bridge',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()


---
## Example 19: Treemap

**Discussion points**

- The treemap encodes part-to-whole relationships using nested rectangles whose area represents proportion. It is effective for large numbers of parts with significant size range across two levels of hierarchy.
- The outer grouping is by division (color family); the inner grouping is by department (individual rectangles).
- Area is a low-accuracy encoding channel. The treemap reads as a pattern rather than a precise quantitative comparison. For precise comparisons, a bar chart is more accurate.
- Values should be labeled directly on each rectangle when space permits.
- The treemap is not appropriate when the number of categories is small enough for a bar chart to work.

In [ ]:
divs={'Operations':{'Logistics':145,'Fulfillment':210,'Quality Control':88},
      'Technology':{'Engineering':320,'Data Science':95,'IT Support':72},
      'Commercial':{'Sales':185,'Marketing':110,'Customer Success':98}}
dp={'Operations':['#2166ac','#4393c3','#92c5de'],
    'Technology':['#d6604d','#f4a582','#fddbc7'],
    'Commercial':['#1a9641','#74c476','#c7e9c0']}
lbt=[]; szt=[]; clt=[]
for div,depts in divs.items():
    for j,(dept,cnt) in enumerate(depts.items()):
        lbt.append(f'{dept}\n({cnt})')
        szt.append(cnt)
        clt.append(dp[div][j])
fig,ax=plt.subplots(figsize=(12,7))
squarify.plot(sizes=szt,label=lbt,color=clt,alpha=0.85,ax=ax,text_kwargs={'fontsize':9})
ax.set_title('Company Headcount by Division and Department',fontsize=13)
ax.axis('off')
lp=[mpatches.Patch(color='#2166ac',label='Operations'),
    mpatches.Patch(color='#d6604d',label='Technology'),
    mpatches.Patch(color='#1a9641',label='Commercial')]
ax.legend(handles=lp,loc='lower right',fontsize=10,title='Division')
plt.tight_layout(); plt.show()


---
## Example 21: Pie Chart -- Legitimate and Illegitimate Use

**Discussion points**

- The left panel shows a legitimate use: three segments with proportions sufficiently distinct (58%, 29%, 13%) to be discriminable by angle and area.
- The right panel shows an illegitimate use: six segments ranging from 11% to 22%. Rank order cannot be reliably determined from visual inspection alone.
- The defensible rule: no more than five segments; proportions must differ meaningfully; the message is specifically about the composition of a single whole.
- When these conditions are not met, a horizontal bar chart sorted by value is more accurate and requires less interpretive effort.
- The goal is not to prohibit pie charts categorically. The goal is to use them only when their encoding channel is adequate for the judgment the audience must make.

In [ ]:
lp1=['Market Leader','Second Tier','All Others']
sp1=[58,29,13]; cp1=['#2166ac','#74add1','#e0f3f8']
lp2=['Seg. A','Seg. B','Seg. C','Seg. D','Seg. E','Seg. F']
sp2=[22,19,18,16,14,11]
cp2=['#2166ac','#4dac26','#d01c8b','#f1a340','#998ec3','#d73027']
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].pie(sp1,labels=lp1,colors=cp1,autopct='%1.0f%%',startangle=90,textprops={'fontsize':11})
axes[0].set_title('Appropriate Use: 3 Segments, Distinct Proportions',fontsize=11)
axes[1].pie(sp2,labels=lp2,colors=cp2,autopct='%1.0f%%',startangle=90,textprops={'fontsize':9})
axes[1].set_title('Inappropriate Use: 6 Segments, Similar Proportions\n(Rank order is ambiguous)',fontsize=11)
plt.suptitle('Example 21: Pie Chart -- When It Works and When It Does Not',fontsize=13)
plt.tight_layout(); plt.show()
